In [1]:
import pandas as pd
from tqdm import tqdm  # opcional: progresso
import dns.resolver
from ipwhois import IPWhois
import os

BRASIL_DOMAINS = r'..\financial_data\brazil_companies_full_data.csv'

UE_DOMAINS = r'..\financial_data\eu_companies_full_data.csv'


br_pd = pd.read_csv(BRASIL_DOMAINS, sep=';')
br_dns_pipeline_pd = br_pd[['domain']]

eu_pd = pd.read_csv(UE_DOMAINS, sep=';')
eu_dns_pipeline_pd = eu_pd[['domain']]

import dns.resolver
from ipwhois import IPWhois


def extract_org_name(rdap_result):
    """
    Extração robusta do nome da organização a partir da resposta RDAP.

    """
    for obj in rdap_result.get("objects", {}).values():
        contact = obj.get("contact", {})
        roles = obj.get("roles", [])
        if "registrant" in roles or contact.get("kind") == "group":
            name = contact.get("name")
            if name:
                return name.replace('"', '').replace("'", "")
    return None

import dns.resolver
from ipwhois import IPWhois
import os

def get_ns_info(domain):
    domain = str(domain)
    ns_data = []

    try:
        # 1. Obter todos os NSs
        ns_answers = dns.resolver.resolve(domain, 'NS')

        for ns in ns_answers:
            ns_name = str(ns.target).rstrip('.')

            # 2. Obter A e AAAA do NS
            ns_ip = None
            ns_ipv6 = None
            try:
                a_answers = dns.resolver.resolve(ns_name, 'A')
                ns_ip = a_answers[0].to_text()
            except:
                pass

            try:
                aaaa_answers = dns.resolver.resolve(ns_name, 'AAAA')
                ns_ipv6 = aaaa_answers[0].to_text()
            except:
                pass

            # 3. Obter ASN e informações com IPWhois
            ip_to_use = ns_ip or ns_ipv6
            if ip_to_use:
                try:
                    whois_obj = IPWhois(ip_to_use)
                    result = whois_obj.lookup_rdap()
                    asn = result.get("asn")
                    as_name = result.get("asn_description").split(',')[0] if result.get("asn_description") else None
                    as_source = result.get("asn_registry")
                    as_country = result.get("asn_country_code")
                    org_name = extract_org_name(result)
                except Exception as e:
                    asn = as_name = as_source = as_country = org_name = None
            else:
                asn = as_name = as_source = as_country = org_name = None

            ns_data.append({
                "NS": ns_name,
                "NS-A": ns_ip,
                "NS-AAAA": ns_ipv6,
                "ASN": asn,
                "ASName": as_name,
                "ASSource": as_source,
                "ASCountry": as_country,
                "OrgName": org_name
            })

        return ns_data

    except Exception as e:
        domain_trimmed = str(domain).split('.', 1)[-1] if '.' in domain else domain
        print(f"Error processing domain {domain}: {e}")

        if domain_trimmed != domain:
            print(f"Trying trimmed domain: {domain_trimmed}")
            return get_ns_info(str(domain_trimmed))
        else:
            return [{"NS": None, "NS-A": None, "NS-AAAA": None, "ASN": None, "ASName": None, "ASSource": None, "ASCountry": None, "OrgName": None, "error": str(e)}]

        
br_dns_pipeline_pd.head()

,domain
0,petrobras.com.br
1,jbs.com.br
2,bb.com.br
3,raizen.com.br
4,itau.com.br


In [2]:
import swifter
import time

OUTPUT_CSV = "br_dns_pipeline_multi_ns.csv"

INITIAL_INDEX = 0 # Número de domínios já processados anteriormente no caso de interrupção do script ou reinício do processamento.

# Configurações
BATCH_SIZE = 50
SLEEP_BETWEEN_BATCHES = 1  # segundos, se quiser pausar entre lotes
SAVE_MODE = 'a'  # append no CSV

# Carregar dataset
total_domains = len(br_dns_pipeline_pd)

# Cabeçalho para salvar apenas no primeiro write
write_header = True

if os.path.exists(OUTPUT_CSV):
    output_pd = pd.read_csv(OUTPUT_CSV)
else:
    output_pd = pd.DataFrame(columns=['domain'])

display(output_pd.tail())
# Processamento em lotes
for i in range(INITIAL_INDEX, total_domains, BATCH_SIZE):
    batch_df = br_dns_pipeline_pd.iloc[i:i+BATCH_SIZE].copy()

    print(f"\nProcessando domínios {i+1} a {min(i+BATCH_SIZE, total_domains)} de {total_domains}...\n")

    batch_results = []

    for idx, domain in tqdm(enumerate(batch_df['domain']), total=len(batch_df), desc=f"Lote {i//BATCH_SIZE + 1}"):
        # check if domain already processed
        if domain in output_pd['domain'].values:
            print(f"Domínio {domain} já processado. Pulando...")
        else:
            print(f"Processando domínio: {domain}")
            ns_info_list = get_ns_info(domain)

            for ns_info in ns_info_list:
                enriched_row = batch_df.iloc[idx].to_dict()
                enriched_row.update(ns_info)
                batch_results.append(enriched_row)

    # Converter para DataFrame
    batch_results_df = pd.DataFrame(batch_results)

    # Salvar incrementalmente
    batch_results_df.to_csv(OUTPUT_CSV, mode=SAVE_MODE, index=False, header=write_header)

    # Desligar header após o primeiro write
    write_header = False
    print(f"✅ Lote {i//BATCH_SIZE + 1} salvo. Total acumulado: {i + BATCH_SIZE} domínios (aproximadamente).\n")

    # Pausa opcional
    time.sleep(SLEEP_BETWEEN_BATCHES)


,domain,NS,NS-A,NS-AAAA,ASN,ASName,ASSource,ASCountry,OrgName
3371,econodata.com.br,ns-cloud-a2.googledomains.com,216.239.34.106,2001:4860:4802:34::6a,15169,GOOGLE,arin,US,Google LLC
3372,bancotricury.com.br,deck.ns.gocache.com.br,144.22.157.238,NaN,31898,ORACLE-BMC-31898,lacnic,CR,Oracle Corporation
3373,bancotricury.com.br,jet.ns.gocache.com.br,170.82.172.1,NaN,266444,3L CLOUD INTERNET SERVICES LTDA - EPP,lacnic,BR,3L CLOUD INTERNET SERVICES LTDA - EPP
3374,casadopaodequeijo.com.br,e.sec.dns.br,200.192.232.14,2001:12f8:b:1::14,11284,Nucleo de Inf. e Coord. do Ponto BR - NIC.,lacnic,BR,Núcleo de Inf. e Coord. do Ponto BR - NIC.BR
3375,casadopaodequeijo.com.br,d.sec.dns.br,200.160.0.14,2001:12ff::14,22548,Nucleo de Inf. e Coord. do Ponto BR - NIC.,lacnic,BR,Núcleo de Inf. e Coord. do Ponto BR - NIC.BR



Processando domínios 1 a 50 de 1000...



Lote 1: 100%|██████████| 50/50 [00:00<00:00, 24085.82it/s]

Domínio petrobras.com.br já processado. Pulando...
Domínio jbs.com.br já processado. Pulando...
Domínio bb.com.br já processado. Pulando...
Domínio raizen.com.br já processado. Pulando...
Domínio itau.com.br já processado. Pulando...
Domínio raizen.com.br já processado. Pulando...
Domínio banco.bradesco já processado. Pulando...
Domínio saladeimprensa.vale.com já processado. Pulando...
Domínio itau.com.br já processado. Pulando...
Domínio caixa.gov.br já processado. Pulando...
Domínio vibraenergia.com.br já processado. Pulando...
Domínio marfrig.com.br já processado. Pulando...
Domínio cargill.com.br já processado. Pulando...
Domínio ultra.com.br já processado. Pulando...
Domínio cargill.com.br já processado. Pulando...
Domínio santander.com.br já processado. Pulando...
Domínio atacadao.com.br já processado. Pulando...
Domínio bunge.com.br já processado. Pulando...
Domínio ambev.com.br já processado. Pulando...
Domínio raizen.com.br já processado. Pulando...
Domínio btgpactual.com já p


Processando domínios 51 a 100 de 1000...



Lote 2: 100%|██████████| 50/50 [00:00<00:00, 25260.80it/s]

... (119 outputs left out)

Lote 20:  96%|█████████▌| 48/50 [03:56<00:07,  3.51s/it]

Domínio eletrobras.com já processado. Pulando...
Processando domínio: virgo.inc


Lote 20: 100%|██████████| 50/50 [03:59<00:00,  4.80s/it]


✅ Lote 20 salvo. Total acumulado: 1000 domínios (aproximadamente).



In [ ]:
import pandas as pd
import swifter
import time
import os

OUTPUT_CSV = "eu_dns_pipeline_multi_ns.csv"

INITIAL_INDEX = 0 # Número de domínios já processados anteriormente no caso de interrupção do script ou reinício do processamento.

# Configurações
BATCH_SIZE = 50
SLEEP_BETWEEN_BATCHES = 1  # segundos, se quiser pausar entre lotes
SAVE_MODE = 'a'  # append no CSV

# Carregar dataset
total_domains = len(eu_dns_pipeline_pd)

# Cabeçalho para salvar apenas no primeiro write
write_header = True

if os.path.exists(OUTPUT_CSV):
    output_pd = pd.read_csv(OUTPUT_CSV)
else:
    output_pd = pd.DataFrame(columns=['domain'])

# Processamento em lotes
for i in range(INITIAL_INDEX, total_domains, BATCH_SIZE):
    batch_df = eu_dns_pipeline_pd.iloc[i:i+BATCH_SIZE].copy()

    print(f"\nProcessando domínios {i+1} a {min(i+BATCH_SIZE, total_domains)} de {total_domains}...\n")

    batch_results = []

    for idx, domain in tqdm(enumerate(batch_df['domain']), total=len(batch_df), desc=f"Lote {i//BATCH_SIZE + 1}"):
        # check if domain already processed
        if domain in output_pd['domain'].values:
            print(f"Domínio {domain} já processado. Pulando...")
            continue
        ns_info_list = get_ns_info(domain)

        for ns_info in ns_info_list:
            enriched_row = batch_df.iloc[idx].to_dict()
            enriched_row.update(ns_info)
            batch_results.append(enriched_row)

    # Converter para DataFrame
    batch_results_df = pd.DataFrame(batch_results)

    # Salvar incrementalmente
    batch_results_df.to_csv(OUTPUT_CSV, mode=SAVE_MODE, index=False, header=write_header)

    # Desligar header após o primeiro write
    write_header = False
    print(f"✅ Lote {i//BATCH_SIZE + 1} salvo. Total acumulado: {i + BATCH_SIZE} domínios (aproximadamente).\n")

    # Pausa opcional
    time.sleep(SLEEP_BETWEEN_BATCHES)



Processando domínios 1 a 50 de 2608...



Lote 1: 100%|██████████| 50/50 [00:00<00:00, 13502.14it/s]

Domínio vw.com já processado. Pulando...
Domínio shell.com já processado. Pulando...
Domínio glencore.com já processado. Pulando...
Domínio exor.com já processado. Pulando...
Domínio totalenergies.com já processado. Pulando...
Domínio bp.com já processado. Pulando...
Domínio telekom.com já processado. Pulando...
Domínio mbusa.com já processado. Pulando...
Domínio bmwusa.com já processado. Pulando...
Domínio axa.com já processado. Pulando...
Domínio allianz.com já processado. Pulando...
Domínio prudential.com já processado. Pulando...
Domínio equinor.com já processado. Pulando...
Domínio nestle.com já processado. Pulando...
Domínio stellantis.com já processado. Pulando...
Domínio aholddelhaize.com já processado. Pulando...
Domínio eni.com já processado. Pulando...
Domínio group.dhl.com já processado. Pulando...
Domínio carrefour.com já processado. Pulando...
Domínio snb.ch já processado. Pulando...
Domínio eon.com já processado. Pulando...
Domínio lvmh.com já processado. Pulando...
Domí


Processando domínios 51 a 100 de 2608...



Lote 2: 100%|██████████| 50/50 [00:00<00:00, 15713.71it/s]

Domínio daimlertruck.com já processado. Pulando...
Domínio chubb.com já processado. Pulando...
Domínio maersk.com já processado. Pulando...
Domínio astrazeneca.com já processado. Pulando...
Domínio riotinto.com já processado. Pulando...
Domínio novartis.com já processado. Pulando...
Domínio bayer.com já processado. Pulando...
Domínio traton.com já processado. Pulando...
Domínio volvogroup.com já processado. Pulando...
Domínio saint-gobain.com já processado. Pulando...
Domínio sanofi.com já processado. Pulando...
Domínio veolia.com já processado. Pulando...
Domínio ubs.com já processado. Pulando...
Domínio loreal.com já processado. Pulando...
Domínio orange.com já processado. Pulando...
Domínio munichre.com já processado. Pulando...
Domínio novocare.com já processado. Pulando...
Domínio swissre.com já processado. Pulando...
Domínio telefonica.com já processado. Pulando...
Domínio imperialbrandsplc.com já processado. Pulando...
Domínio porsche.com já processado. Pulando...
Domínio compas

... (171 outputs left out)


Processando domínios 2601 a 2608 de 2608...



Lote 53: 100%|██████████| 8/8 [00:00<00:00, 10459.61it/s]

Domínio sgt-capital.com já processado. Pulando...
Domínio elbstein.com já processado. Pulando...
Domínio nanobiotix.com já processado. Pulando...
Domínio greencoat-ukwind.com já processado. Pulando...
Domínio ipgroupplc.com já processado. Pulando...
Domínio kinnevik.com já processado. Pulando...
Domínio nbb.be já processado. Pulando...
Domínio porsche-se.com já processado. Pulando...
✅ Lote 53 salvo. Total acumulado: 2650 domínios (aproximadamente).



: 